In [110]:
from transformers import pipeline
from rapidfuzz import fuzz

# Fast toxicity model
toxicity_model = pipeline(
    "text-classification",
    model="unitary/toxic-bert"
)

# Zero-shot model (multi-label in ONE call)
zero_shot_model = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6149.69it/s]
BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 7185.85it/s]


In [111]:
import re

# def normalize_text(text: str) -> str:
#     text = text.lower()

#     replacements = {
#         '1': 'i', '3': 'e', '4': 'a', '5': 's',
#         '0': 'o', '@': 'a', '$': 's', '!': 'i'
#     }
#     for k, v in replacements.items():
#         text = text.replace(k, v)

#     text = re.sub(r'[^a-z0-9\s]', '', text)

#     text = re.sub(
#         r'(\b[a-z]\s+){2,}[a-z]\b',
#         lambda m: m.group(0).replace(" ", ""),
#         text
#     )

#     text = re.sub(r'\s+', ' ', text).strip()

#     return text

In [112]:
def normalize_leetspeak(text: str) -> str:
    replacements = {
        '3': 'e',
        '4': 'a',
        '5': 's',
        '0': 'o',
        '@': 'a',
        '$': 's'
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    # Special handling for 1 → could be i or l
    text = text.replace('1', 'l')  # better for "kill"

    return text

In [113]:
def normalize_text(text: str) -> str:
    text = text.lower()

    text = normalize_leetspeak(text)

    text = re.sub(r'[^a-z0-9\s]', '', text)

    text = re.sub(
        r'(\b[a-z]\s+){2,}[a-z]\b',
        lambda m: m.group(0).replace(" ", ""),
        text
    )

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [114]:
BANNED_PATTERNS = [
    r'k\s*i\s*l\s*l',
    r'k+i+l+l+',
    r'kill',
    r'suicide',
    r'nude',
    r'rape'
]
# BANNED_PATTERNS = []

def rule_check(text: str):
    for pattern in BANNED_PATTERNS:
        if re.search(pattern, text):
            return True, pattern
    return False, None

In [115]:
# def fuzzy_check(text):
#     targets = ["kill", "suicide", "rape", "nude"]
#     # targets = []

#     for word in targets:
#         if fuzz.partial_ratio(text, word) > 85:
#             return True, word

#     return False, None

In [116]:
from rapidfuzz import fuzz
def direct_match(text):
    targets = ["kill", "suicide", "rape", "nude"]

    for word in targets:
        if word in text:
            return True, word

    return False, None

    
def fuzzy_check(text):
    targets = ["kill", "suicide", "rape", "nude"]

    for word in targets:
        score = max(
            fuzz.partial_ratio(text, word),
            fuzz.token_sort_ratio(text, word),
            fuzz.ratio(text, word)
        )

        if score > 80:
            return True, word

    return False, None

In [117]:
def detect_toxicity(text: str) -> float:
    result = toxicity_model(text)[0]

    label = result["label"].lower()
    score = result["score"]

    if "toxic" in label:
        return score
    else:
        return 1 - score

In [118]:
CANDIDATE_LABELS = [
    "sexual content",
    "hate speech",
    "self harm",
    "normal conversation"
]

def detect_multi_labels(text: str) -> dict:
    result = zero_shot_model(text, CANDIDATE_LABELS)

    scores = dict(zip(result["labels"], result["scores"]))

    return {
        "sexual": scores.get("sexual content", 0.0),
        "hate": scores.get("hate speech", 0.0),
        "self_harm": scores.get("self harm", 0.0)
    }

In [119]:
def get_scores(text: str) -> dict:
    toxicity = detect_toxicity(text)
    multi = detect_multi_labels(text)

    return {
        "toxicity": toxicity,
        "sexual": multi["sexual"],
        "hate": multi["hate"],
        "self_harm": multi["self_harm"]
    }

In [120]:
THRESHOLDS = {
    "toxicity": 0.9,
    "sexual": 0.8,
    "hate": 0.8,
    "self_harm": 0.85
}

def decide(scores: dict):
    for label, threshold in THRESHOLDS.items():
        if scores.get(label, 0) > threshold:
            return {"status": "BLOCKED", "reason": label}

    # softer layer
    if scores.get("toxicity", 0) > 0.6:
        return {"status": "FLAGGED", "reason": "toxicity"}

    return {"status": "APPROVED", "reason": None}

In [121]:
# def moderate(text: str):
#     processed = normalize_text(text)
#     # after normalization 
#     fuzzy_hit, word = fuzzy_check(processed)

#     if fuzzy_hit:
#         return {
#             "original": text,
#             "processed": processed,
#             "status": "BLOCKED",
#             "reason": "fuzzy_match",
#             "matched_word": word
#         }
#     # Rule check
#     rule_hit, pattern = rule_check(text)
#     if not rule_hit:
#         rule_hit, pattern = rule_check(processed)

#     if rule_hit:
#         return {
#             "original": text,
#             "processed": processed,
#             "status": "BLOCKED",
#             "reason": "rule_match",
#             "pattern": pattern
#         }

#     # ML
#     scores = get_scores(processed)

#     # Decision
#     decision = decide(scores)

#     return {
#         "original": text,
#         "processed": processed,
#         "scores": scores,
#         "status": decision["status"],
#         "reason": decision["reason"]
#     }

In [122]:
def moderate(text: str):
    processed = normalize_text(text)

    # 1. Direct match
    hit, word = direct_match(processed)
    if hit:
        return {
            "original": text,
            "processed": processed,
            "status": "BLOCKED",
            "reason": "direct_match",
            "matched_word": word
        }

    # 2. Fuzzy match
    fuzzy_hit, word = fuzzy_check(processed)
    if fuzzy_hit:
        return {
            "original": text,
            "processed": processed,
            "status": "BLOCKED",
            "reason": "fuzzy_match",
            "matched_word": word
        }

    # 3. ML model
    scores = get_scores(processed)

    decision = decide(scores)

    return {
        "original": text,
        "processed": processed,
        "scores": scores,
        "status": decision["status"],
        "reason": decision["reason"]
    }

In [124]:
tests = [
    "Hello, how are you?",
    "I hate you",
    "You are stupid",
    "Send me nudes",
    "I want to kill myself",
    "Let's build something amazing",
    "I will kill you",
    "k i l l",
    "k!ll",
    "ki11"
]

# tests = [

#     # -------------------------
#     # ✅ Normal / Safe
#     # -------------------------
#     "Hello, how are you?",
#     "Let's build something amazing",
#     "Good morning!",
#     "I love programming",
#     "See you tomorrow",
#     "That was a great session",
#     "Happy birthday!",
#     "Let's collaborate on this idea",

#     # -------------------------
#     # ⚠️ Mild Toxicity (Should FLAG)
#     # -------------------------
#     "You are stupid",
#     "This is dumb",
#     "I don't like you",
#     "That was a bad idea",
#     "You're annoying",
#     "Stop being useless",

#     # -------------------------
#     # 🚫 Strong Toxicity (Should BLOCK)
#     # -------------------------
#     "I hate you",
#     "You are an idiot",
#     "Go to hell",
#     "You're worthless",
#     "Nobody likes you",
#     "Shut up you idiot",

#     # -------------------------
#     # 🔞 Sexual Content
#     # -------------------------
#     "Send me nudes",
#     "Let's have sex",
#     "Show me your body",
#     "I want to see you naked",
#     "Do you watch porn?",
#     "Let's do something naughty",

#     # -------------------------
#     # ⚠️ Self Harm / Violence
#     # -------------------------
#     "I want to kill myself",
#     "I feel like ending my life",
#     "I will kill you",
#     "You should die",
#     "I want to hurt someone",
#     "Life is not worth living",

#     # -------------------------
#     # 🧠 Obfuscated / Bypass Attempts
#     # -------------------------
#     "k i l l",
#     "k!ll",
#     "ki11",
#     "k1ll",
#     "k.i.l.l",
#     "k-i-l-l",
#     "klll",
#     "kll",
#     "su1cide",
#     "n@de",
#     "ra.pe",

#     # -------------------------
#     # 😈 Clever Bypass (Realistic)
#     # -------------------------
#     "go ki11 yourself",
#     "you should k i l l yourself",
#     "send n*des",
#     "n u d e pics please",
#     "I h@te you",
#     "you are st*pid",
#     "shut the f*** up",

#     # -------------------------
#     # 🧪 Edge Cases
#     # -------------------------
#     "kill time",
#     "this task will kill the process",
#     "execute the process",
#     "class assignment",
#     "pass the exam",
#     "assistant",
#     "analysis of system",
# ]

for t in tests:
    print(moderate(t))
    print("-" * 60)

{'original': 'Hello, how are you?', 'processed': 'hello how are you', 'scores': {'toxicity': 0.001277025556191802, 'sexual': 0.10954323410987854, 'hate': 0.05218954384326935, 'self_harm': 0.06209038943052292}, 'status': 'APPROVED', 'reason': None}
------------------------------------------------------------
{'original': 'I hate you', 'processed': 'i hate you', 'scores': {'toxicity': 0.9509986639022827, 'sexual': 0.046628788113594055, 'hate': 0.9325507879257202, 'self_harm': 0.016311481595039368}, 'status': 'BLOCKED', 'reason': 'toxicity'}
------------------------------------------------------------
{'original': 'You are stupid', 'processed': 'you are stupid', 'scores': {'toxicity': 0.9847338795661926, 'sexual': 0.24534021317958832, 'hate': 0.6147755980491638, 'self_harm': 0.09052154421806335}, 'status': 'BLOCKED', 'reason': 'toxicity'}
------------------------------------------------------------
{'original': 'Send me nudes', 'processed': 'send me nudes', 'status': 'BLOCKED', 'reason': 